In [ ]:
import cuvis
import time
import io
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from PIL import Image as PILImage

print("Initializing Cuvis")
cuvis.General.init("./settings")
camera_serial_number_str = "YourCameraSerialNumber"

calib = cuvis.SessionFile(F"./factory/{camera_serial_number_str}.cu3c")
acq_cont = cuvis.AcquisitionContext(calib)

while(not acq_cont.ready):
    time.sleep(1)
    print(".", end="")
print("\nCamera connected!")
time.sleep(0.5)

acq_cont.set_continuous(False)  # Pause video stream for now
acq_cont.operation_mode = cuvis.OperationMode.Internal
acq_cont.integration_time = 100
acq_cont.fps = 2

# Create ProcessingContext
proc_cont = cuvis.ProcessingContext(calib)
proc_cont.processing_mode = cuvis.ProcessingMode.Raw

# Create CubeExporter and viewer
viewer = cuvis.Viewer(cuvis.ViewerSettings(userplugin="./plugins/00_RGB.xml", pan_scale=1.0))

## Helper Function
def show_next_frame(frame, fmt='jpeg', jpeg_quality=85):
    arr = np.asarray(frame)
    if arr.ndim == 2:
        arr = np.repeat(arr[..., None], 3, axis=2)
    if arr.shape[-1] == 4:
        arr = arr[..., :3]
    if arr.dtype != np.uint8:
        arr = np.clip(arr, 0, 255).astype(np.uint8)

    buf = io.BytesIO()
    if fmt == 'jpeg':
        PILImage.fromarray(arr).save(buf, format='JPEG', quality=jpeg_quality)
    else:
        PILImage.fromarray(arr).save(buf, format='PNG')
    img.value = buf.getvalue()


## Show image in notebook
img = widgets.Image(format='jpeg')
img.layout = widgets.Layout(border='1px solid #ddd', width='auto')
display(img)

print("Acquiring...")
acq_cont.set_continuous(True)
while True:
    if acq_cont.has_next_measurement():
        result = acq_cont.get_next_measurement(100)
        print(result.data.keys())
        proc_cont.apply(result)
        views = viewer.apply(result)
        show_next_frame(views.array)
    else:
        time.sleep(0.1)